Install Libraries

In [1]:
!pip install datasets audiomentations -q
from datasets import load_dataset, DatasetDict, Dataset
from datasets.features import Audio
from sklearn.model_selection import train_test_split
import pandas as pd
import audiomentations as A
from tqdm import tqdm

Load Dataset

In [2]:
dataset = load_dataset("PolyAI/minds14", "en-US")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
        num_rows: 563
    })
})

Cek Struktur Data

In [3]:
print("\n-------Struktur Data-----------")
print(f"Kolom: {dataset['train'].column_names}")
print(f"Jumlah Data Train Asli: {len(dataset['train'])}")
print(f"\n Contoh Data:")
print( dataset['train'][0])


-------Struktur Data-----------
Kolom: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id']
Jumlah Data Train Asli: 563

 Contoh Data:
{'path': 'en-US~JOINT_ACCOUNT/602ba55abb1e6d0fbce92065.wav', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7f16cd8f8ec0>, 'transcription': 'I would like to set up a joint account with my partner', 'english_transcription': 'I would like to set up a joint account with my partner', 'intent_class': 11, 'lang_id': 4}


Hapus Kolom yang tidak digunakan

In [4]:
columns_to_remove = ['lang_id', 'english_transcription', 'path']
dataset = dataset.remove_columns(columns_to_remove)
dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 563
    })
})

Split Data

In [5]:
full_data = dataset['train']
indices=list(range(len(full_data)))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=full_data['intent_class'])
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=[full_data['intent_class'][i] for i in temp_idx])

#Buat Dataset Splits
train_original = full_data.select(train_idx)
val_original = full_data.select(val_idx)
test_original = full_data.select(test_idx)

Resample dan Augment data train

In [6]:
# Resample

train_original_augmented = train_original.cast_column("audio", Audio(sampling_rate=16000))

In [7]:
#Augment

#Definisi
augmenter = A.Compose([
    A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    A.PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    A.TimeStretch(min_rate=0.8, max_rate=1.25, p=0.5),
])

# Initialize empty lists to store augmented data
augmented_audios_data = [] # This will store dicts: {'array': ..., 'sampling_rate': ...}
augmented_intents = []
augmented_transcripts = []

for sample in tqdm(train_original_augmented):
  audio_array = sample['audio']['array']
  sr = sample['audio']['sampling_rate']
  intent = sample['intent_class']
  transcript = sample['transcription']

  # Add original sample
  augmented_audios_data.append({'array': audio_array, 'sampling_rate': sr})
  augmented_intents.append(intent)
  augmented_transcripts.append(transcript)

  # Add 3 augmented versions per original sample
  for _ in range(3):
    augmented_audio_array = augmenter(samples=audio_array, sample_rate=sr)
    augmented_audios_data.append({'array': augmented_audio_array, 'sampling_rate': sr})
    augmented_intents.append(intent)
    augmented_transcripts.append(transcript)

# Create the new dataset from the augmented data
train_augmented = Dataset.from_dict({
    'audio': augmented_audios_data,
    'intent_class': augmented_intents,
    'transcription': augmented_transcripts
})

print(f"Selesai! Jumlah data train sebelum augmentasi: {len(train_original_augmented) // 4} -> setelah augmentasi: {len(train_augmented)}")

100%|██████████| 450/450 [01:12<00:00,  6.19it/s]


Selesai! Jumlah data train sebelum augmentasi: 112 -> setelah augmentasi: 1800


In [13]:
'''from datasets import Audio

#Resample Train

train_augmented = train_augmented.cast_column("audio", Audio(sampling_rate=16000))

AttributeError: 'list' object has no attribute 'T'

Resample Val & Test (Tanpa Augmentasi)

In [8]:
val_resampled = val_original.cast_column("audio", Audio(sampling_rate=16000))
test_resampled = test_original.cast_column("audio", Audio(sampling_rate=16000))

In [9]:
for i in range(5):
  print(len(train_augmented[i]['audio']['array']))
  print(len(val_resampled[i]['audio']['array']))
  print(len(test_resampled[i]['audio']['array']))

682666
62806
152918
682666
228010
61440
682666
141994
77824
682666
105130
27306
460118
69846
125610


In [2]:
train_augmented = Dataset.from_dict({
    "audio": [x["audio"]["array"] for x in train_augmented],
    "intent_class": [x["intent_class"] for x in train_augmented],
    "transcription": [x["transcription"] for x in train_augmented]
})

NameError: name 'Dataset' is not defined

In [ ]:
#Gabungkan ke DatasetDict
dataset_split_minds14 = DatasetDict({
    'train': train_augmented,
    'validation': val_resampled,
    'test': test_resampled
})

dataset_split_minds14

## Simpan Dataset

In [ ]:
save_dir = "/content/drive/MyDrive/Colab Notebooks/data project 3 (Voice Recognition)/minds14_augmented"
dataset_split_minds14.save_to_disk(save_dir)

Simpan metadata untuk referensi

In [ ]:
import json
from datetime import datetime

metadata = {
    "dataset_name": "minds14_enUS_augmented",
    "created_at": datetime.now().isoformat(),
    "original_samples": len(train_original),
    "augmented_train_samples": len(train_augmented),
    "validation_samples": len(val_resampled),
    "test_samples": len(test_resampled),
    "augmentation_factor": 4,  # 1 original + 3 augmented
    "sampling_rate": 16000,
    "split_seed": 42,
    "columns": ['audio', 'transcription', 'intent_class'],
    "augmentation_applied_only_to_train": True,
    "augmentation_methods": ["AddGaussianNoise", "PitchShift", "TimeStretch"]
}

with open(f"{save_dir}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

In [ ]:
# Verifikasi Save
from datasets import load_from_disk

print("\n🔍 Verifying saved dataset...")
loaded_back = load_from_disk(save_dir)
print(f"✓ Load successful!")
print(f"  Train: {len(loaded_back['train'])} samples")
print(f"  Validation: {len(loaded_back['validation'])} samples")
print(f"  Test: {len(loaded_back['test'])} samples")

In [ ]:
#Zip Untuk Download
import shutil
zip_path = f"{save_dir}.zip"
print(f"\n Creating zip archive: {zip_path}")
shutil.make_archive(save_dir, 'zip', save_dir)
print(f" Zip created: {zip_path}")

print("\n All done! Dataset ready to use.")